# Clustering Metrics Validation

This notebook demonstrates and validates the `colliderml.clustering` metrics suite.
Each section introduces a concept with a **toy example**, then applies it to **real ColliderML data**.

## Metrics Overview

| Metric | What it answers |
|--------|----------------|
| **Efficiency** | Were truth particles found? (energy-weighted) |
| **Purity** | Are reco clusters real? (energy-weighted) |
| **Energy resolution** | How accurate is E measurement? (σ_eff) |
| **Splitting rate** | Over-segmentation |
| **Merging rate** | Under-segmentation |
| **Fake rate** | Fraction of reco clusters without truth match |
| **V-score** | Cell-assignment quality (information-theoretic) |

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from colliderml.clustering import (
    build_truth_clusters,
    build_overlap_matrix,
    greedy_match,
    evaluate_clustering,
    weighted_v_score,
    sigma_eff,
)

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(20)
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 12})

---
## 1. Truth Cluster Construction

A **truth cluster** groups all calorimeter energy from a single primary ancestor particle.
Since showers overlap, a cell can belong to multiple truth clusters with different energy fractions.

### 1a. Toy Example

Set up a tiny event: 2 ancestor particles, 4 descendant particles, 5 calorimeter cells.
Cell 2 is shared — it receives energy from both ancestors.

In [ ]:
# Decay chain: ancestor A (id=10) -> descendants 11, 12
#              ancestor B (id=20) -> descendants 21, 22
particles_toy = pl.DataFrame({
    "event_id": [1],
    "particle_id": [[10, 11, 12, 20, 21, 22]],
    "parent_id":   [[-1, 10, 10, -1, 20, 20]],
    "energy":      [[100., 60., 40., 80., 50., 30.]],
    "px": [[10., 6., 4., 8., 5., 3.]],
    "py": [[0.]*6],
    "pz": [[0.]*6],
})

# 5 cells, cell 2 is shared between the two showers
calo_toy = pl.DataFrame({
    "event_id": [1],
    "detector":    [[10, 10, 10, 13, 13]],
    "x":           [[0., 1., 2., 3., 4.]],
    "y":           [[0., 0., 0., 0., 0.]],
    "z":           [[0., 0., 0., 0., 0.]],
    "total_energy": [[8., 5., 6., 4., 7.]],
    "contrib_particle_ids": [[[11], [12], [12, 21], [21], [22]]],
    "contrib_energies":     [[[8.], [5.], [3., 3.], [4.], [7.]]],
    "contrib_times":        [[[0.1], [0.1], [0.1, 0.2], [0.2], [0.2]]],
})

tc_toy = build_truth_clusters(calo_toy, particles_toy, apply_calibration=False)
tc_toy

In [ ]:
# Visualize: which cells belong to which ancestor
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
for anc_id, color in [(10, "tab:blue"), (20, "tab:orange")]:
    subset = tc_toy[tc_toy["primary_ancestor_id"] == anc_id]
    ax.bar(
        subset["cell_index"], subset["energy"],
        bottom=0 if anc_id == 10 else tc_toy[tc_toy["primary_ancestor_id"] == 10].set_index("cell_index").reindex(subset["cell_index"])["energy"].fillna(0).values,
        label=f"Ancestor {anc_id}", color=color, alpha=0.8, edgecolor="k"
    )
ax.set_xlabel("Cell index")
ax.set_ylabel("Energy [GeV]")
ax.set_title("Toy: soft truth clusters (cell 2 is shared)")
ax.legend()
plt.tight_layout()
plt.show()

### 1b. Real Data

In [ ]:
from colliderml.core import load_tables, collect_tables

cfg = {
    "dataset_id": "CERN/ColliderML-Release-1",
    "channels": "ttbar",
    "pileup": "pu0",
    "objects": ["particles", "calo_hits"],
    "split": "train",
    "lazy": False,
    "max_events": 10,
    "data_dir": "/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_lib/data",
}
tables = load_tables(cfg)
frames = collect_tables(tables)

In [ ]:
tc_real = build_truth_clusters(frames["calo_hits"], frames["particles"])
print(f"Truth cluster rows: {len(tc_real):,}")
print(f"Unique (event, ancestor) pairs: {tc_real.groupby(['event_id', 'primary_ancestor_id']).ngroups:,}")
print(f"Unique events: {tc_real['event_id'].nunique()}")
tc_real.head(10)

In [ ]:
# Distribution: how many truth clusters per event, and cells per truth cluster
clusters_per_event = tc_real.groupby("event_id")["primary_ancestor_id"].nunique()
cells_per_cluster = tc_real.groupby(["event_id", "primary_ancestor_id"])["cell_index"].nunique()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(clusters_per_event, bins=30, edgecolor="k", alpha=0.7)
ax1.set_xlabel("Truth clusters per event")
ax1.set_ylabel("Count")
ax1.set_title(f"Truth clusters/event (median={clusters_per_event.median():.0f})")

ax2.hist(cells_per_cluster, bins=50, edgecolor="k", alpha=0.7, range=(0, 200))
ax2.set_xlabel("Cells per truth cluster")
ax2.set_ylabel("Count")
ax2.set_title(f"Cells/cluster (median={cells_per_cluster.median():.0f})")
plt.tight_layout()
plt.show()

---
## 2. Overlap Matrix & EIOU

Given truth clusters and predicted labels, the **overlap matrix** `O[t, r]` measures how much
energy from truth cluster `t` ended up in reco cluster `r`.

**EIOU** (Energy-weighted IoU) = `O[t,r] / (E_truth[t] + E_reco[r] - O[t,r])`

### 2a. Toy: Perfect, Merged, and Split Clustering

In [ ]:
# Perfect: each truth cluster maps to exactly one reco cluster
pred_perfect = pd.DataFrame({
    "event_id": [1, 1, 1, 1, 1],
    "cell_index": [0, 1, 2, 3, 4],
    "cluster_id": [0, 0, 0, 1, 1],  # cells 0-2 → cluster A, cells 3-4 → cluster B
})

# Merged: everything in one reco cluster
pred_merged = pred_perfect.copy()
pred_merged["cluster_id"] = 0

# Split: ancestor A's cells split across two reco clusters
pred_split = pred_perfect.copy()
pred_split["cluster_id"] = [0, 1, 1, 2, 2]  # cell 0 alone, cells 1-2 with 3-4

for name, pred in [("Perfect", pred_perfect), ("Merged", pred_merged), ("Split", pred_split)]:
    overlaps = build_overlap_matrix(tc_toy, pred)
    print(f"\n--- {name} ---")
    print(overlaps[["primary_ancestor_id", "cluster_id", "overlap_energy", "eiou"]].to_string(index=False))

### 2b. Toy: Overlap Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

for ax, (name, pred) in zip(axes, [("Perfect", pred_perfect), ("Merged", pred_merged), ("Split", pred_split)]):
    overlaps = build_overlap_matrix(tc_toy, pred)
    pivot = overlaps.pivot_table(index="primary_ancestor_id", columns="cluster_id", values="eiou", fill_value=0)
    im = ax.imshow(pivot.values, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"R{int(c)}" for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"T{int(t)}" for t in pivot.index])
    ax.set_xlabel("Reco cluster")
    ax.set_ylabel("Truth cluster")
    ax.set_title(f"{name}")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i,j]:.2f}", ha="center", va="center", fontsize=10)

fig.colorbar(im, ax=axes, label="EIOU", shrink=0.8)
fig.suptitle("EIOU Overlap Heatmaps", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Greedy Matching

Match truth ↔ reco clusters greedily by descending EIOU. Each side matched at most once.

In [ ]:
# Show matching for the merged case — only one truth can claim the single reco cluster
overlaps_merged = build_overlap_matrix(tc_toy, pred_merged)
matches_merged = greedy_match(overlaps_merged, threshold=0.3)
print("Merged case matching:")
print(matches_merged.to_string(index=False))

In [ ]:
# Show matching for the split case
overlaps_split = build_overlap_matrix(tc_toy, pred_split)
matches_split = greedy_match(overlaps_split, threshold=0.3)
print("Split case matching:")
print(matches_split.to_string(index=False))

---
## 4. Full Metrics — Noise Sweep

Start from perfect clustering, then randomly reassign increasing fractions of cells.
All metrics should degrade smoothly.

In [ ]:
# Build truth labels for real data (dominant ancestor per cell)
dominant_real = (
    tc_real.sort_values("energy", ascending=False)
    .groupby(["event_id", "cell_index"], as_index=False)
    .first()[["event_id", "cell_index", "primary_ancestor_id", "total_energy"]]
)

# Perfect pred: cluster_id = primary_ancestor_id
pred_real_perfect = dominant_real[["event_id", "cell_index", "primary_ancestor_id"]].rename(
    columns={"primary_ancestor_id": "cluster_id"}
)

rng = np.random.default_rng(42)
noise_fracs = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
sweep_results = []

unique_clusters = pred_real_perfect["cluster_id"].unique()

for frac in noise_fracs:
    pred_noisy = pred_real_perfect.copy()
    n_flip = int(len(pred_noisy) * frac)
    if n_flip > 0:
        flip_idx = rng.choice(len(pred_noisy), size=n_flip, replace=False)
        pred_noisy.iloc[flip_idx, pred_noisy.columns.get_loc("cluster_id")] = rng.choice(
            unique_clusters, size=n_flip
        )
    result = evaluate_clustering(tc_real, pred_noisy)
    result["noise_frac"] = frac
    sweep_results.append(result)
    print(f"noise={frac:.0%}: eff={result['efficiency']:.3f}  pur={result['purity']:.3f}  "
          f"Eres={result['energy_resolution']:.4f}  V={result['v_score']:.3f}  "
          f"fake={result['fake_rate']:.3f}  split={result['splitting_rate']:.3f}  merge={result['merging_rate']:.3f}")

In [ ]:
# Plot all metrics vs noise
fracs = [r["noise_frac"] for r in sweep_results]
metrics_to_plot = [
    ("efficiency", "Efficiency"),
    ("purity", "Purity"),
    ("v_score", "V-score"),
    ("fake_rate", "Fake rate"),
    ("splitting_rate", "Splitting rate"),
    ("merging_rate", "Merging rate"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for ax, (key, label) in zip(axes.flat, metrics_to_plot):
    vals = [r[key] for r in sweep_results]
    ax.plot(fracs, vals, "o-", lw=2)
    ax.set_ylabel(label)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

for ax in axes[1]:
    ax.set_xlabel("Fraction of cells randomly reassigned")

fig.suptitle("Metrics vs. random cell reassignment noise (ttbar pu0, 10 events)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. V-score: Per-Event vs Merged-Event

The ML Power Week 2023 paper found that computing V-score across all events merged gives
artificially high scores. Here we demonstrate this.

In [ ]:
# Compute V-score two ways: per-event (correct) and merged (inflated)
merged_truth = dominant_real["primary_ancestor_id"].to_numpy()
merged_pred = pred_real_perfect["cluster_id"].to_numpy()
merged_weights = dominant_real["total_energy"].to_numpy().astype(float)

# Merged across all events (wrong)
_, _, v_merged = weighted_v_score(merged_truth, merged_pred, merged_weights)

# Per-event then averaged (correct — what evaluate_clustering does)
result_perfect = evaluate_clustering(tc_real, pred_real_perfect)
v_per_event = result_perfect["v_score"]

print(f"V-score (merged across events): {v_merged:.4f}")
print(f"V-score (per-event averaged):   {v_per_event:.4f}")
print(f"\nThe merged score is {'higher' if v_merged > v_per_event else 'comparable'} — "
      f"particle IDs from different events create artificial cluster separation.")

---
## 6. Energy Weighting Matters

Demonstrate a case where count-based and energy-weighted metrics disagree.
Many low-energy cells misclustered, but high-energy core correct.

In [ ]:
# Build a toy event: one truth cluster with 5 high-energy core cells and 95 low-energy halo cells
n_core, n_halo = 5, 95
truth_demo = pd.DataFrame({
    "event_id": [1] * (n_core + n_halo),
    "cell_index": list(range(n_core + n_halo)),
    "primary_ancestor_id": [10] * (n_core + n_halo),
    "energy": [100.0] * n_core + [0.1] * n_halo,
    "total_energy": [100.0] * n_core + [0.1] * n_halo,
})

# Predicted: core cells correct (cluster 0), halo cells scattered across random clusters
pred_demo = pd.DataFrame({
    "event_id": [1] * (n_core + n_halo),
    "cell_index": list(range(n_core + n_halo)),
    "cluster_id": [0] * n_core + list(rng.integers(1, 50, size=n_halo)),
})

result_demo = evaluate_clustering(truth_demo, pred_demo)

# Count-based: only 5/100 cells are in the "right" cluster → low
# Energy-weighted: 500/509.5 GeV is correctly assigned → high
count_purity = n_core / (n_core + n_halo)
energy_correct = n_core * 100.0
energy_total = n_core * 100.0 + n_halo * 0.1

print(f"Count-based 'purity' (cells): {count_purity:.1%}")
print(f"Energy-weighted efficiency:    {result_demo['efficiency']:.1%}")
print(f"Energy-weighted purity:        {result_demo['purity']:.1%}")
print(f"\nEnergy weighting correctly reflects that the physics-relevant")
print(f"core is well-reconstructed despite many noise cells being scattered.")

---
## 7. sigma_eff: Robust Resolution Estimator

In [ ]:
# Compare sigma_eff to standard deviation for Gaussian and non-Gaussian
rng2 = np.random.default_rng(123)

# Gaussian
gauss = rng2.normal(1.0, 0.1, size=10000)
# Gaussian + long tail (10% of entries)
tail = np.concatenate([rng2.normal(1.0, 0.1, size=9000), rng2.normal(1.0, 0.5, size=1000)])

for name, data in [("Gaussian (σ=0.1)", gauss), ("Gaussian + tail", tail)]:
    se = sigma_eff(data)
    sd = np.std(data)
    print(f"{name:25s}: sigma_eff={se:.4f}, std={sd:.4f}, ratio={se/sd:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, data) in zip([ax1, ax2], [("Gaussian", gauss), ("Gaussian + tail", tail)]):
    ax.hist(data, bins=80, range=(0.5, 1.5), alpha=0.7, edgecolor="k")
    se = sigma_eff(data)
    sd = np.std(data)
    ax.axvline(np.median(data) - se, color="r", ls="--", label=f"sigma_eff={se:.4f}")
    ax.axvline(np.median(data) + se, color="r", ls="--")
    ax.axvline(np.mean(data) - sd, color="b", ls=":", label=f"std={sd:.4f}")
    ax.axvline(np.mean(data) + sd, color="b", ls=":")
    ax.set_title(name)
    ax.legend()
    ax.set_xlabel("E_reco / E_truth")
plt.suptitle("sigma_eff is robust to non-Gaussian tails", fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Real Data: KMeans Baseline

Evaluate simple KMeans clustering on real ColliderML data to see how the metrics behave.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from colliderml.polars import explode_calo_cells_and_contribs
from colliderml.physics.calibration import apply_calo_calibration, odd_default_calo_calibration

# Get cell positions for clustering
calo_cal = apply_calo_calibration(frames["calo_hits"], odd_default_calo_calibration())
cells_df, _ = explode_calo_cells_and_contribs(calo_cal)

k_values = [20, 50, 100, 200, 500]
kmeans_results = []

for k in k_values:
    pred_rows = []
    for eid, event_cells in cells_df.groupby("event_id"):
        coords = event_cells[["x", "y", "z"]].values.astype(np.float32)
        n_clusters = min(k, len(coords))
        km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1000)
        labels = km.fit_predict(coords)
        event_pred = event_cells[["event_id", "cell_index"]].copy()
        event_pred["cluster_id"] = labels
        pred_rows.append(event_pred)
    
    pred_kmeans = pd.concat(pred_rows, ignore_index=True)
    result = evaluate_clustering(tc_real, pred_kmeans)
    result["k"] = k
    kmeans_results.append(result)
    print(f"k={k:4d}: eff={result['efficiency']:.3f}  pur={result['purity']:.3f}  "
          f"Eres={result['energy_resolution']:.4f}  V={result['v_score']:.3f}  "
          f"split={result['splitting_rate']:.3f}  merge={result['merging_rate']:.3f}")

In [ ]:
# Plot metrics vs k
ks = [r["k"] for r in kmeans_results]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for ax, (key, label) in zip(axes.flat, metrics_to_plot):
    vals = [r[key] for r in kmeans_results]
    ax.plot(ks, vals, "o-", lw=2)
    ax.set_ylabel(label)
    ax.set_xscale("log")
    ax.grid(True, alpha=0.3)

for ax in axes[1]:
    ax.set_xlabel("Number of KMeans clusters (k)")

fig.suptitle("KMeans clustering metrics vs k (ttbar pu0, 10 events)", fontsize=14)
plt.tight_layout()
plt.show()